<a href="https://colab.research.google.com/github/Hafizh213/Modul-2-AI/blob/upstream/Modul2Kelompok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from google.colab import files

# ==============================================================================
# 1. BACA DATASET AWAL & INSPEKSI
# ==============================================================================
df = pd.read_csv('dataset_penjualan_kantin.csv')

print("--- 5 Baris Pertama ---")
print(df.head(5))

print("\n--- Info Dataset ---")
df.info()

print("\n--- Jumlah Missing Value ---")
print(df.isnull().sum())

print(f"\n--- Jumlah Duplikat: {df.duplicated().sum()} ---")


# ==============================================================================
# 2. FUNGSI KHUSUS PEMBERSIH TANGGAL
# ==============================================================================
def bersihkan_tanggal(val):
    if pd.isna(val):
        return val
    val_str = str(val).strip()

    bulan_dict = {
        'januari': '01', 'februari': '02', 'maret': '03', 'april': '04',
        'mei': '05', 'juni': '06', 'juli': '07', 'agustus': '08',
        'september': '09', 'oktober': '10', 'november': '11', 'desember': '12'
    }

    # Format "14 Agustus 2026"
    for bln_nama, bln_angka in bulan_dict.items():
        if bln_nama in val_str.lower():
            bagian = val_str.split()
            if len(bagian) == 3:
                tgl = bagian[0].zfill(2)
                thn = bagian[2]
                return f"{thn}-{bln_angka}-{tgl}"

    # Format "13/08/2026"
    if '/' in val_str:
        bagian = val_str.split('/')
        if len(bagian) == 3:
            tgl = bagian[0].zfill(2)
            bln = bagian[1].zfill(2)
            thn = bagian[2]
            return f"{thn}-{bln}-{tgl}"

    return val_str


# ==============================================================================
# 3. PROSES PEMBERSIHAN DATASET
# ==============================================================================
df_clean = df.copy()

# A. Hapus Duplikat
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

# B. Standarisasi Teks Kategori
df_clean['kategori'] = df_clean['kategori'].str.capitalize()

# C. Bersihkan Format Harga Satuan
df_clean['harga_satuan'] = df_clean['harga_satuan'].astype(str).str.replace('Rp', '', regex=False)
df_clean['harga_satuan'] = df_clean['harga_satuan'].str.replace('.', '', regex=False).str.strip()
df_clean['harga_satuan'] = pd.to_numeric(df_clean['harga_satuan'], errors='coerce')

harga_produk_map = {
    'Nasi Goreng': 12000, 'Mie Ayam': 10000, 'Bakso': 11000,
    'Soto Ayam': 12000, 'Gorengan': 2000, 'Roti Bakar': 7000,
    'Kerupuk': 2000, 'Es Teh': 3000, 'Es Jeruk': 4000,
    'Jus Alpukat': 8000, 'Kopi': 5000, 'Air Mineral': 3000
}
df_clean['harga_satuan'] = df_clean['harga_satuan'].fillna(df_clean['nama_produk'].map(harga_produk_map))

# D. Bersihkan Jumlah Terjual
df_clean['jumlah_terjual'] = df_clean['jumlah_terjual'].astype(str).str.replace('pcs', '', case=False, regex=False).str.strip()
df_clean['jumlah_terjual'] = pd.to_numeric(df_clean['jumlah_terjual'], errors='coerce')
df_clean['jumlah_terjual'] = df_clean['jumlah_terjual'].fillna(df_clean['jumlah_terjual'].median())

# E. Bersihkan Tanggal & Kasir
df_clean['tanggal'] = df_clean['tanggal'].ffill().bfill()
df_clean['tanggal'] = df_clean['tanggal'].apply(bersihkan_tanggal)
df_clean['nama_kasir'] = df_clean['nama_kasir'].fillna(df_clean['nama_kasir'].mode()[0])

# F. Contoh Menggunakan NumPy: Mengubah tipe data ke np.int64 & Menghitung Total
df_clean['jumlah_terjual'] = df_clean['jumlah_terjual'].astype(np.int64)
df_clean['harga_satuan'] = df_clean['harga_satuan'].astype(np.int64)
df_clean['total_penjualan'] = np.multiply(df_clean['jumlah_terjual'], df_clean['harga_satuan'])


# ==============================================================================
# 4. RINGKASAN OUTPUT TERAKHIR
# ==============================================================================
ringkasan_kategori = df_clean.groupby('kategori')[['jumlah_terjual', 'total_penjualan']].sum().reset_index()

print("\n--- Ringkasan Penjualan per Kategori ---")
print(ringkasan_kategori)


# ==============================================================================
# 5. SIMPAN DATASET BERSIH
# ==============================================================================
df_clean.to_csv('dataset_bersih.csv', index=False)
print("\nDataset bersih berhasil disimpan ke 'dataset_bersih.csv'")

try:
    files.download('dataset_bersih.csv')
except Exception:
    pass

FileNotFoundError: [Errno 2] No such file or directory: 'dataset_penjualan_kantin.csv'